# Feature Engineering

**Project:** Predicting Performance Trajectories of Women Micro-Enterprises: An Early Warning Framework

This notebook prepares candidate predictor features for the modelling stage.

### Main feature groups

1. Socio-demographic features
2. Baseline business features
3. Engagement and attendance features
4. Behavior Point features
5. Baseline digital-readiness features
6. SSM / business formalisation features
7. Pre/Post competency delta features



In [2]:
# 0. Setup

from google.colab import drive
from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/24088950_MAI_Research"
)

DATA_PATH = (
    PROJECT_DIR
    / "Ibupreneur_Academy_Microenterprise_Performance_Raw_Dataset_2025_2026.xlsm"
)

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RELATIONSHIP_TARGET_PATH = (
    OUTPUT_DIR
    / "relationship_base_with_target.csv"
)

TARGET_PATH = (
    OUTPUT_DIR
    / "trajectory_target.csv"
)

print("Project folder:", PROJECT_DIR)
print("Raw dataset:", DATA_PATH)
print("Output folder:", OUTPUT_DIR)
print("Target-linked dataset:", RELATIONSHIP_TARGET_PATH)

print("\nRaw dataset exists:", DATA_PATH.exists())
print("Target-linked dataset exists:", RELATIONSHIP_TARGET_PATH.exists())
print("Target file exists:", TARGET_PATH.exists())

Mounted at /content/drive
Project folder: /content/drive/MyDrive/24088950_MAI_Research
Raw dataset: /content/drive/MyDrive/24088950_MAI_Research/Ibupreneur_Academy_Microenterprise_Performance_Raw_Dataset_2025_2026.xlsm
Output folder: /content/drive/MyDrive/24088950_MAI_Research/outputs
Target-linked dataset: /content/drive/MyDrive/24088950_MAI_Research/outputs/relationship_base_with_target.csv

Raw dataset exists: True
Target-linked dataset exists: True
Target file exists: True


## 1. Load Notebook 02 Outputs and Raw Supporting Sheets


In [3]:
# 1. Load connected outputs

if not RELATIONSHIP_TARGET_PATH.exists():
    raise FileNotFoundError(
        "relationship_base_with_target.csv was not found. "
        "Run Notebook 02 and save its outputs first."
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset was not found: {DATA_PATH}"
    )

base = pd.read_csv(
    RELATIONSHIP_TARGET_PATH
)

target = pd.read_csv(
    TARGET_PATH
)

print("Base shape:", base.shape)
print("Target shape:", target.shape)

display(base.head())

Base shape: (201, 81)
Target shape: (173, 3)


,Cohort ID,Funder/Program,Year,Cohort Label,Participant ID,Status,Full Name [REMOVED - PII],IC Number [REMOVED - PII],State,Age,...,delta_comfortable_working_and_sharing_ideas_in_a_team,delta_confident_with_product_packaging_appearance,delta_confident_in_the_quality_of_the_photos_you_share_for_business,delta_confident_in_the_quality_of_the_videos_you_share_for_business,delta_confident_to_deliver_an_elevator_pitch,delta_confident_in_managing_businees_finance,delta_understand_the_main_purpose_of_a_business_plan,delta_confident_i_can_prepare_a_complete_and_convincing_business_plan,delta_confident_speaking_in_front_of_audiences,Performance Trajectory
0,MIB-2025-C1,MIB,2025,Cohort 1 (Stay-In),2025_IBU-ACADEMY-MIB-C1-01,GRADUATED,NaN,NaN,SABAH,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,High
1,MIB-2025-C1,MIB,2025,Cohort 1 (Stay-In),2025_IBU-ACADEMY-MIB-C1-02,GRADUATED,NaN,NaN,JOHOR,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,High
2,MIB-2025-C1,MIB,2025,Cohort 1 (Stay-In),2025_IBU-ACADEMY-MIB-C1-03,GRADUATED,NaN,NaN,MELAKA,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MIB-2025-C1,MIB,2025,Cohort 1 (Stay-In),2025_IBU-ACADEMY-MIB-C1-04,GRADUATED,NaN,NaN,SELANGOR,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Moderate
4,MIB-2025-C1,MIB,2025,Cohort 1 (Stay-In),2025_IBU-ACADEMY-MIB-C1-05,GRADUATED,NaN,NaN,NEGERI SEMBILAN,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Moderate


In [4]:
print('Listing contents of the project directory:')
for item in PROJECT_DIR.iterdir():
    print(item.name)

Listing contents of the project directory:
Ibupreneur_Academy_Microenterprise_Performance_Raw_Dataset_2025_2026.xlsm
outputs
01_EDA_Data_Understanding.ipynb
02_Target_Construction.ipynb
04_Final_Modelling_Dataset.ipynb
05_All_Models_and_Comparison.ipynb
03_Feature_Engineering.ipynb


In [5]:
# 1.1 Load raw feature sheets

dem = pd.read_excel(
    DATA_PATH,
    sheet_name="Demographics",
    header=3
)

prepost = pd.read_excel(
    DATA_PATH,
    sheet_name="Pre-Post Module",
    header=3
)

finance = pd.read_excel(
    DATA_PATH,
    sheet_name="Finance Sales Profit",
    header=3
)

ssm = pd.read_excel(
    DATA_PATH,
    sheet_name="SSM",
    header=3
)

attendance = pd.read_excel(
    DATA_PATH,
    sheet_name="Attendance Modules",
    header=3
)

behavior = pd.read_excel(
    DATA_PATH,
    sheet_name="Behavior Points",
    header=3
)

social = pd.read_excel(
    DATA_PATH,
    sheet_name="Social Media",
    header=4
)

social = social.iloc[:, :17].copy()

social.columns = [
    "Cohort ID",
    "Funder/Program",
    "Year",
    "Participant ID",
    "Full Name [REMOVED - PII]",
    "IG_has_pre",
    "IG_followers_pre",
    "FB_has_pre",
    "FB_followers_pre",
    "TT_has_pre",
    "TT_followers_pre",
    "IG_has_post",
    "IG_followers_post",
    "FB_has_post",
    "FB_followers_post",
    "TT_has_post",
    "TT_followers_post",
]

for df in [
    dem,
    prepost,
    finance,
    ssm,
    attendance,
    behavior,
    social
]:
    df.dropna(
        subset=["Participant ID"],
        inplace=True
    )

print("Raw sheets loaded.")

Raw sheets loaded.


## 2. Restrict Feature Engineering to Participants with a Valid Target


In [6]:
# 2. Keep only participants with a valid target

model_base = base[
    base["Performance Trajectory"].notna()
].copy()

valid_target_ids = set(
    model_base["Participant ID"]
)

print(
    "Participants with valid target:",
    model_base["Participant ID"].nunique()
)

print(
    model_base[
        "Performance Trajectory"
    ].value_counts()
)

Participants with valid target: 173
Performance Trajectory
Moderate    84
High        46
Low         43
Name: count, dtype: int64


## 3. Leakage Guard


In [7]:
# 3. Define known leakage / outcome variables

leakage_keywords = [
    "sales month 1",
    "sales month 2",
    "sales month 3",
    "total sales",
    "average monthly sales",
    "sales growth",
    "profit month 1",
    "profit month 2",
    "profit month 3",
    "total profit",
    "profit growth",
    "post",
    "latest",
    "bonus payout",
    "performance trajectory"
]

known_leakage_columns = []

for column in base.columns:

    lower = str(column).lower()

    if any(
        keyword in lower
        for keyword in leakage_keywords
    ):
        known_leakage_columns.append(
            column
        )

print("Potential leakage columns detected:")

for column in known_leakage_columns:
    print("-", column)

Potential leakage columns detected:
- Performance Trajectory


## 4. Socio-Demographic Features


In [8]:
# 4. Prepare socio-demographic features

demographic_candidates = [
    "Participant ID",
    "Cohort ID",
    "Funder/Program",
    "Year",
    "Age",
    "State",
    "Marital Status",
    "Education Level",
    "Working Status",
    "Industry/Business Sector",
    "Years in Business",
    "Asnaf Status",
    "Household Income",
    "Total Commitment",
    "Estimated Remaining Balance"
]

demographic_candidates = [
    column
    for column in demographic_candidates
    if column in dem.columns
]

dem_features = (
    dem[
        demographic_candidates
    ]
    .drop_duplicates(
        "Participant ID"
    )
    .copy()
)

numeric_demographic = [
    "Age",
    "Years in Business",
    "Household Income",
    "Total Commitment",
    "Estimated Remaining Balance"
]

for column in numeric_demographic:

    if column in dem_features.columns:

        dem_features[column] = pd.to_numeric(
            dem_features[column],
            errors="coerce"
        )

dem_features = dem_features[
    dem_features["Participant ID"].isin(
        valid_target_ids
    )
]

display(
    dem_features.head()
)

,Participant ID,Cohort ID,Funder/Program,Year,Age,State,Marital Status,Education Level,Working Status,Industry/Business Sector,Years in Business,Asnaf Status,Household Income,Total Commitment,Estimated Remaining Balance
0,2025_IBU-ACADEMY-MIB-C1-01,MIB-2025-C1,MIB,2025,26,SABAH,Married,Diploma,Self-Employed,F&B,12.0,B40,7500.0,7907.0,-407.0
1,2025_IBU-ACADEMY-MIB-C1-02,MIB-2025-C1,MIB,2025,26,JOHOR,Married,SPM,Self-Employed,F&B,2.0,FAKIR,1050.0,1069.0,931.0
3,2025_IBU-ACADEMY-MIB-C1-04,MIB-2025-C1,MIB,2025,26,SELANGOR,Divorced,SPM,Self-Employed,F&B,2.0,B40,1500.0,2093.0,-593.0
4,2025_IBU-ACADEMY-MIB-C1-05,MIB-2025-C1,MIB,2025,26,NEGERI SEMBILAN,Married,Diploma,Self-Employed,F&B,12.0,B40,2500.0,3191.0,-391.0
5,2025_IBU-ACADEMY-MIB-C1-06,MIB-2025-C1,MIB,2025,26,KUALA LUMPUR,Divorced,SPM,Self-Employed,F&B,10.0,MISKIN,1500.0,2354.0,-129.0


## 5. Baseline Business Features

Only variables known before the outcome period are retained.

`Sales Before Program` and `Profit Before Program` are baseline business-condition features.

Month 1–3 sales and profit are excluded.

In [9]:
# 5. Prepare baseline business features

baseline_business_cols = [
    "Participant ID",
    "Sales Before Program",
    "Profit Before Program"
]

baseline_business = (
    finance[
        baseline_business_cols
    ]
    .drop_duplicates(
        "Participant ID"
    )
    .copy()
)

for column in [
    "Sales Before Program",
    "Profit Before Program"
]:

    baseline_business[column] = pd.to_numeric(
        baseline_business[column],
        errors="coerce"
    )

baseline_business = baseline_business[
    baseline_business["Participant ID"].isin(
        valid_target_ids
    )
]

display(
    baseline_business.head()
)

,Participant ID,Sales Before Program,Profit Before Program
0,2025_IBU-ACADEMY-MIB-C1-01,8800.0,15000.0
1,2025_IBU-ACADEMY-MIB-C1-02,800.0,360.0
3,2025_IBU-ACADEMY-MIB-C1-04,2800.0,2400.0
4,2025_IBU-ACADEMY-MIB-C1-05,2300.0,1500.0
5,2025_IBU-ACADEMY-MIB-C1-06,3500.0,250.0


## 6. Attendance Feature


In [10]:
# 6. Extract attendance rate

def parse_percentage(x):

    if pd.isna(x):
        return np.nan

    if isinstance(
        x,
        (int, float, np.number)
    ):
        value = float(x)

    else:

        text = (
            str(x)
            .strip()
            .replace("%", "")
            .replace(",", "")
        )

        if text.upper() in [
            "",
            "N/A",
            "NA",
            "-",
            "NONE",
            "NAN"
        ]:
            return np.nan

        value = pd.to_numeric(
            text,
            errors="coerce"
        )

    if pd.isna(value):
        return np.nan

    if 0 <= value <= 1:
        value *= 100

    return value


attendance_rate_cols = [
    column
    for column in attendance.columns
    if "attendance rate" in str(column).lower()
]

print("Attendance-rate columns detected:")

for column in attendance_rate_cols:
    print("-", column)

Attendance-rate columns detected:
- MIB: Attendance Rates
- MBSB: Attendance Rates
- AIB-2026: Attendance Rates
- UNDP-2026: Attendance Rates
- CIMB-2026: Attendance Rates


In [11]:
# 6.1 Build participant-level attendance feature

attendance_temp = attendance[
    ["Participant ID"] + attendance_rate_cols
].copy()

for column in attendance_rate_cols:

    attendance_temp[column] = (
        attendance_temp[column]
        .map(parse_percentage)
    )

if attendance_rate_cols:

    attendance_temp["attendance_rate"] = (
        attendance_temp[
            attendance_rate_cols
        ]
        .bfill(axis=1)
        .iloc[:, 0]
    )

else:

    attendance_temp[
        "attendance_rate"
    ] = np.nan


attendance_features = (
    attendance_temp[
        [
            "Participant ID",
            "attendance_rate"
        ]
    ]
    .drop_duplicates(
        "Participant ID"
    )
)

attendance_features = attendance_features[
    attendance_features["Participant ID"].isin(
        valid_target_ids
    )
]

display(
    attendance_features.head()
)

print(
    "Attendance coverage:",
    attendance_features["attendance_rate"].notna().mean() * 100
)

,Participant ID,attendance_rate
0,2025_IBU-ACADEMY-MIB-C1-01,76.9
1,2025_IBU-ACADEMY-MIB-C1-02,76.9
3,2025_IBU-ACADEMY-MIB-C1-04,76.9
4,2025_IBU-ACADEMY-MIB-C1-05,76.9
5,2025_IBU-ACADEMY-MIB-C1-06,84.6


Attendance coverage: 100.0


## 7. Behavior Point Features

Behavior Points are recorded at module/session level.

The modelling features are aggregated to one row per participant.

Because cohorts have different numbers of modules/sessions, both raw and within-cohort normalized behavior features are created.

In [12]:
# 7. Convert behavior columns to numeric

behavior_numeric_cols = [
    "Module Order",
    "Start Point",
    "Gain",
    "Deduct",
    "Net Behavior Point",
    "Running Total",
    "Final Behavior Points",
    "Final Behavior Score %"
]

for column in behavior_numeric_cols:

    if column in behavior.columns:

        behavior[column] = pd.to_numeric(
            behavior[column],
            errors="coerce"
        )

In [13]:
# 7.1 Aggregate Behavior Points to participant level

def last_non_null(series):

    values = series.dropna()

    if len(values) == 0:
        return np.nan

    return values.iloc[-1]


behavior_summary = (
    behavior
    .sort_values(
        [
            "Participant ID",
            "Module Order"
        ]
    )
    .groupby(
        [
            "Participant ID",
            "Cohort ID"
        ],
        as_index=False
    )
    .agg(
        behavior_sessions=(
            "Module / Session",
            "count"
        ),
        behavior_total_gain=(
            "Gain",
            "sum"
        ),
        behavior_total_deduct=(
            "Deduct",
            "sum"
        ),
        behavior_net_total=(
            "Net Behavior Point",
            "sum"
        ),
        behavior_avg_gain_per_session=(
            "Gain",
            "mean"
        ),
        behavior_avg_deduct_per_session=(
            "Deduct",
            "mean"
        ),
        behavior_net_std=(
            "Net Behavior Point",
            "std"
        ),
        behavior_final_points=(
            "Final Behavior Points",
            last_non_null
        )
    )
)

display(
    behavior_summary.head()
)

,Participant ID,Cohort ID,behavior_sessions,behavior_total_gain,behavior_total_deduct,behavior_net_total,behavior_avg_gain_per_session,behavior_avg_deduct_per_session,behavior_net_std,behavior_final_points
0,2025_IBU-ACADEMY-MBSB-C1-02,MBSB-2025-C1,8,44.0,0.0,44.0,5.500,NaN,4.140393,94.0
1,2025_IBU-ACADEMY-MBSB-C1-03,MBSB-2025-C1,8,45.0,15.0,30.0,5.625,15.0,8.048957,80.0
2,2025_IBU-ACADEMY-MBSB-C1-04,MBSB-2025-C1,8,59.0,0.0,59.0,7.375,NaN,3.113909,109.0
3,2025_IBU-ACADEMY-MBSB-C1-05,MBSB-2025-C1,8,59.0,0.0,59.0,7.375,NaN,3.113909,109.0
4,2025_IBU-ACADEMY-MBSB-C1-06,MBSB-2025-C1,8,61.0,0.0,61.0,7.625,NaN,3.248626,111.0


In [14]:
# 7.2 Create within-cohort normalized behavior features

behavior_summary[
    "behavior_final_percentile_in_cohort"
] = (
    behavior_summary
    .groupby("Cohort ID")[
        "behavior_final_points"
    ]
    .rank(
        pct=True,
        method="average"
    )
)


def within_group_z(series):

    sd = series.std(ddof=0)

    if pd.isna(sd) or sd == 0:

        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (
        series - series.mean()
    ) / sd


behavior_summary[
    "behavior_final_z_in_cohort"
] = (
    behavior_summary
    .groupby("Cohort ID")[
        "behavior_final_points"
    ]
    .transform(
        within_group_z
    )
)


behavior_features = behavior_summary[
    [
        "Participant ID",
        "behavior_sessions",
        "behavior_total_gain",
        "behavior_total_deduct",
        "behavior_net_total",
        "behavior_avg_gain_per_session",
        "behavior_avg_deduct_per_session",
        "behavior_net_std",
        "behavior_final_points",
        "behavior_final_percentile_in_cohort",
        "behavior_final_z_in_cohort"
    ]
].copy()

behavior_features = behavior_features[
    behavior_features["Participant ID"].isin(
        valid_target_ids
    )
]

display(
    behavior_features.head()
)

,Participant ID,behavior_sessions,behavior_total_gain,behavior_total_deduct,behavior_net_total,behavior_avg_gain_per_session,behavior_avg_deduct_per_session,behavior_net_std,behavior_final_points,behavior_final_percentile_in_cohort,behavior_final_z_in_cohort
0,2025_IBU-ACADEMY-MBSB-C1-02,8,44.0,0.0,44.0,5.500,NaN,4.140393,94.0,0.425,-0.354240
1,2025_IBU-ACADEMY-MBSB-C1-03,8,45.0,15.0,30.0,5.625,15.0,8.048957,80.0,0.050,-1.823682
3,2025_IBU-ACADEMY-MBSB-C1-05,8,59.0,0.0,59.0,7.375,NaN,3.113909,109.0,0.900,1.220161
4,2025_IBU-ACADEMY-MBSB-C1-06,8,61.0,0.0,61.0,7.625,NaN,3.248626,111.0,1.000,1.430082
5,2025_IBU-ACADEMY-MBSB-C1-07,8,59.0,0.0,59.0,7.375,NaN,3.113909,109.0,0.900,1.220161


## 8. Baseline Digital-Readiness Features

Only baseline social-media variables are used.

Post-program follower counts are excluded because follow-up is incomplete and they may occur after the predictor cutoff.

In [15]:
# 8. Parse follower counts

def parse_count(x):

    if pd.isna(x):
        return np.nan

    text = (
        str(x)
        .strip()
        .upper()
        .replace(",", "")
    )

    if text in [
        "",
        "N/A",
        "NA",
        "NONE",
        "-",
        "NAN"
    ]:
        return np.nan

    match = re.fullmatch(
        r"([0-9]*\.?[0-9]+)\s*([KM]?)\+?",
        text
    )

    if not match:

        return pd.to_numeric(
            text,
            errors="coerce"
        )

    value = float(
        match.group(1)
    )

    suffix = match.group(2)

    if suffix == "K":
        value *= 1000

    elif suffix == "M":
        value *= 1000000

    return value


for column in [
    "IG_followers_pre",
    "FB_followers_pre",
    "TT_followers_pre"
]:

    social[column] = social[column].map(
        parse_count
    )

In [16]:
# 8.1 Convert baseline platform adoption to binary

def yes_no_to_binary(x):

    if pd.isna(x):
        return np.nan

    value = str(x).strip().lower()

    yes_values = [
        "yes",
        "ya",
        "y",
        "1",
        "true",
        "ada"
    ]

    no_values = [
        "no",
        "tidak",
        "n",
        "0",
        "false",
        "tiada"
    ]

    if value in yes_values:
        return 1

    if value in no_values:
        return 0

    return np.nan


for column in [
    "IG_has_pre",
    "FB_has_pre",
    "TT_has_pre"
]:

    social[
        f"{column}_binary"
    ] = social[column].map(
        yes_no_to_binary
    )

In [17]:
# 8.2 Create participant-level digital features

digital_features = social[
    [
        "Participant ID",
        "IG_has_pre_binary",
        "IG_followers_pre",
        "FB_has_pre_binary",
        "FB_followers_pre",
        "TT_has_pre_binary",
        "TT_followers_pre"
    ]
].copy()


digital_features[
    "digital_platform_count"
] = (
    digital_features[
        [
            "IG_has_pre_binary",
            "FB_has_pre_binary",
            "TT_has_pre_binary"
        ]
    ]
    .sum(
        axis=1,
        min_count=1
    )
)


digital_features = (
    digital_features
    .drop_duplicates(
        "Participant ID"
    )
)


digital_features = digital_features[
    digital_features["Participant ID"].isin(
        valid_target_ids
    )
]

display(
    digital_features.head()
)

,Participant ID,IG_has_pre_binary,IG_followers_pre,FB_has_pre_binary,FB_followers_pre,TT_has_pre_binary,TT_followers_pre,digital_platform_count
0,2025_IBU-ACADEMY-MIB-C1-01,1,50.0,1,100.0,1,1000.0,3
1,2025_IBU-ACADEMY-MIB-C1-02,1,0.0,0,NaN,1,4055.0,2
3,2025_IBU-ACADEMY-MIB-C1-04,1,0.0,1,365.0,1,1291.0,3
4,2025_IBU-ACADEMY-MIB-C1-05,0,NaN,1,2400.0,1,628.0,2
5,2025_IBU-ACADEMY-MIB-C1-06,1,0.0,1,67.0,1,227.0,3


## 9. SSM / Business Formalisation Feature

SSM registration is converted to a binary feature only when the source value clearly represents Yes or No.

In [18]:
# 9. Prepare SSM registration feature

ssm_features = ssm[
    [
        "Participant ID",
        "SSM Registered"
    ]
].copy()

ssm_features[
    "ssm_registered_binary"
] = (
    ssm_features[
        "SSM Registered"
    ]
    .map(
        yes_no_to_binary
    )
)

ssm_features = (
    ssm_features[
        [
            "Participant ID",
            "ssm_registered_binary"
        ]
    ]
    .drop_duplicates(
        "Participant ID"
    )
)

ssm_features = ssm_features[
    ssm_features["Participant ID"].isin(
        valid_target_ids
    )
]

display(
    ssm_features.head()
)

,Participant ID,ssm_registered_binary
0,2025_IBU-ACADEMY-MIB-C1-01,1.0
1,2025_IBU-ACADEMY-MIB-C1-02,1.0
3,2025_IBU-ACADEMY-MIB-C1-04,1.0
4,2025_IBU-ACADEMY-MIB-C1-05,1.0
5,2025_IBU-ACADEMY-MIB-C1-06,1.0


## 10. Detect Pre/Post Competency Pairs

This step identifies matching Pre and Post columns based on column names.

The detected mapping should be reviewed before final modelling because some cohort-specific questions may use different wording or scales.

In [19]:
# 10. Detect Pre/Post pairs

def identify_pre_post(
    column_name
):

    name = str(
        column_name
    ).strip()

    lower = name.lower()

    is_pre = bool(
        re.search(
            r"\bpre\b|\(pre\)|pre[-_ ]",
            lower
        )
    )

    is_post = bool(
        re.search(
            r"\bpost\b|\(post\)|post[-_ ]",
            lower
        )
    )

    if (
        not is_pre
        and not is_post
    ):
        return None, None

    base_name = re.sub(
        r"\bpre\b|\bpost\b|\(pre\)|\(post\)|pre[-_ ]|post[-_ ]",
        "",
        lower
    )

    base_name = re.sub(
        r"[^a-z0-9]+",
        " ",
        base_name
    )

    base_name = re.sub(
        r"\s+",
        " ",
        base_name
    ).strip()

    if is_pre:
        return "pre", base_name

    if is_post:
        return "post", base_name

    return None, None


pair_dictionary = {}

for column in prepost.columns:

    period, base_name = (
        identify_pre_post(
            column
        )
    )

    if period is None:
        continue

    if base_name not in pair_dictionary:
        pair_dictionary[
            base_name
        ] = {}

    pair_dictionary[
        base_name
    ][period] = column


pre_post_pairs = []

for base_name, pair in pair_dictionary.items():

    if (
        "pre" in pair
        and "post" in pair
    ):

        pre_post_pairs.append({
            "Construct": base_name,
            "Pre Column": pair["pre"],
            "Post Column": pair["post"]
        })


pre_post_pairs = pd.DataFrame(
    pre_post_pairs
)

print(
    "Potential Pre/Post pairs detected:",
    len(pre_post_pairs)
)

display(
    pre_post_pairs
)

Potential Pre/Post pairs detected: 67


,Construct,Pre Column,Post Column
0,able to identify 3 personal strengths and 3 we...,Pre - Able to identify 3 personal strengths a...,Post - Able to identify 3 personal strengths ...
1,have vision board,Pre - Have vision board,Post - Have vision board
2,your product label does it include the require...,Pre- Your product label — does it include the ...,Post - Your product label — does it include th...
3,have a standard operating procedure sop docume...,Pre - Have a Standard Operating Procedure (SOP...,Post - Have a Standard Operating Procedure (SO...
4,aware actual cost and selling price,Pre - Aware actual cost and selling price,Post - Aware actual cost and selling price
...,...,...,...
62,understand the main purpose of a business plan,Pre - Understand the main purpose of a Busines...,Post - Understand the main purpose of a Busine...
63,confident i can prepare a complete and convinc...,Pre - Confident I can prepare a complete and c...,Post - Confident I can prepare a complete and ...
64,confident speaking in front of audiences,Pre - Confident speaking in front of audiences,Post - Confident speaking in front of audiences
65,have presented your business or idea in front ...,Pre - Have presented your business or idea in ...,Post - Have presented your business or idea in...


## 11. Build Numeric Competency Delta Candidates

For numeric Pre/Post pairs:

**Delta = Post − Pre**

Only pairs with at least 20 valid participants are retained as candidate delta features.

This does not mean every candidate will automatically enter the final model. Notebook 04 will review coverage, redundancy, timing, and leakage.

In [20]:
# 11. Create numeric Pre/Post delta candidates

prepost_target = prepost[
    prepost["Participant ID"].isin(
        valid_target_ids
    )
].copy()


delta_features = prepost_target[
    [
        "Participant ID"
    ]
].copy()


delta_feature_dictionary = []


for _, row in pre_post_pairs.iterrows():

    construct = row[
        "Construct"
    ]

    pre_col = row[
        "Pre Column"
    ]

    post_col = row[
        "Post Column"
    ]

    pre_numeric = pd.to_numeric(
        prepost_target[
            pre_col
        ],
        errors="coerce"
    )

    post_numeric = pd.to_numeric(
        prepost_target[
            post_col
        ],
        errors="coerce"
    )

    valid_pair = (
        pre_numeric.notna()
        & post_numeric.notna()
    )

    valid_n = int(
        valid_pair.sum()
    )

    if valid_n < 20:
        continue

    safe_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        construct.lower()
    ).strip("_")

    delta_name = (
        f"delta_{safe_name}"
    )

    delta_features[
        delta_name
    ] = (
        post_numeric
        - pre_numeric
    )

    delta_feature_dictionary.append({
        "Feature": delta_name,
        "Construct": construct,
        "Pre Column": pre_col,
        "Post Column": post_col,
        "Valid n": valid_n
    })


delta_feature_dictionary = pd.DataFrame(
    delta_feature_dictionary
)


print(
    "Numeric delta candidates created:",
    len(delta_feature_dictionary)
)

display(
    delta_feature_dictionary
)

Numeric delta candidates created: 16


,Feature,Construct,Pre Column,Post Column,Valid n
0,delta_able_to_identify_3_personal_strengths_an...,able to identify 3 personal strengths and 3 we...,Pre - Able to identify 3 personal strengths a...,Post - Able to identify 3 personal strengths ...,25
1,delta_confidence_level_in_your_leadership,confidence level in your leadership,Pre - Confidence level in your leadership,Post - Confidence level in your leadership,36
2,delta_well_know_who_target_customer,well know who target customer,Pre - Well know who target customer,Post - Well know who target customer,23
3,delta_know_the_pain_point_customer_face_that_m...,know the pain point customer face that makes t...,Pre - Know the Pain Point customer face that m...,Post - Know the Pain Point customer face that ...,23
4,delta_i_am_confident_with_the_price_i_set,i am confident with the price i set,Pre - I am confident with the price I set,Post - I am confident with the price I set,23
5,delta_confidence_level_in_pitching_your_busine...,confidence level in pitching your business ide...,Pre - Confidence level in pitching your busine...,Post - Confidence level in pitching your busin...,40
6,delta_level_of_courage_in_answering_spontaneou...,level of courage in answering spontaneous ques...,Pre - Level of courage in answering spontaneou...,Post - Level of courage in answering spontaneo...,23
7,delta_comfortable_working_and_sharing_ideas_in...,comfortable working and sharing ideas in a team,Pre - Comfortable working and sharing ideas in...,Post - Comfortable working and sharing ideas i...,23
8,delta_confident_with_product_packaging_appearance,confident with product packaging appearance,Pre - confident with product packaging appearance,Post - confident with product packaging appear...,25
9,delta_confident_in_the_quality_of_the_photos_y...,confident in the quality of the photos you sha...,Pre - Confident in the quality of the photos y...,Post - Confident in the quality of the photos ...,25


## 12. Delta Feature Quality Check

This section checks:

- coverage;
- mean and median change;
- negative / zero / positive change;
- extreme values.

A negative delta is not automatically an error. It may represent a genuine decline.

In [21]:
# 12. Summarise competency deltas

delta_columns = [
    column
    for column in delta_features.columns
    if column != "Participant ID"
]

delta_summary = []

for column in delta_columns:

    values = delta_features[
        column
    ]

    observed = values.dropna()

    delta_summary.append({
        "Feature": column,
        "Available n": int(
            observed.shape[0]
        ),
        "Missing %": round(
            values.isna().mean() * 100,
            1
        ),
        "Mean Delta": observed.mean(),
        "Median Delta": observed.median(),
        "Negative n": int(
            (observed < 0).sum()
        ),
        "No Change n": int(
            (observed == 0).sum()
        ),
        "Positive n": int(
            (observed > 0).sum()
        ),
        "Min": observed.min(),
        "Max": observed.max()
    })


delta_summary = pd.DataFrame(
    delta_summary
)

display(
    delta_summary
    .sort_values(
        "Available n",
        ascending=False
    )
)

,Feature,Available n,Missing %,Mean Delta,Median Delta,Negative n,No Change n,Positive n,Min,Max
5,delta_confidence_level_in_pitching_your_busine...,40,76.9,1.000000,1.0,2,10,28,-1.0,4.0
1,delta_confidence_level_in_your_leadership,36,79.2,0.888889,0.0,4,15,17,-1.0,4.0
0,delta_able_to_identify_3_personal_strengths_an...,25,85.5,0.560000,1.0,4,8,13,-1.0,3.0
8,delta_confident_with_product_packaging_appearance,25,85.5,1.040000,1.0,0,8,17,0.0,4.0
12,delta_confident_in_managing_businees_finance,25,85.5,1.080000,1.0,1,7,17,-1.0,3.0
10,delta_confident_in_the_quality_of_the_videos_y...,25,85.5,1.160000,1.0,0,6,19,0.0,4.0
9,delta_confident_in_the_quality_of_the_photos_y...,25,85.5,1.200000,1.0,0,4,21,0.0,2.0
2,delta_well_know_who_target_customer,23,86.7,0.782609,1.0,2,9,12,-1.0,3.0
7,delta_comfortable_working_and_sharing_ideas_in...,23,86.7,0.347826,0.0,3,13,7,-1.0,2.0
6,delta_level_of_courage_in_answering_spontaneou...,23,86.7,0.956522,1.0,2,4,17,-1.0,3.0


## 13. Merge Candidate Feature Families

This creates one participant-level engineered-feature table.

`Cohort ID` is retained as metadata for auditing and stratification. It should not automatically be used as a predictor because the model could learn cohort-specific differences rather than participant risk.

In [22]:
# 13. Merge feature families

engineered = model_base[
    [
        "Participant ID",
        "Performance Trajectory"
    ]
].copy()


engineered = engineered.merge(
    dem_features,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)

engineered = engineered.merge(
    baseline_business,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)

engineered = engineered.merge(
    attendance_features,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)

engineered = engineered.merge(
    behavior_features,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)

engineered = engineered.merge(
    digital_features,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)

engineered = engineered.merge(
    ssm_features,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)

engineered = engineered.merge(
    delta_features,
    on="Participant ID",
    how="left",
    validate="one_to_one"
)


print(
    "Engineered dataset shape:",
    engineered.shape
)

print(
    "Unique participants:",
    engineered[
        "Participant ID"
    ].nunique()
)

display(
    engineered.head()
)

Engineered dataset shape: (173, 53)
Unique participants: 173


,Participant ID,Performance Trajectory,Cohort ID,Funder/Program,Year,Age,State,Marital Status,Education Level,Working Status,...,delta_level_of_courage_in_answering_spontaneous_questions_q_a_from_panel_members_or_investors,delta_comfortable_working_and_sharing_ideas_in_a_team,delta_confident_with_product_packaging_appearance,delta_confident_in_the_quality_of_the_photos_you_share_for_business,delta_confident_in_the_quality_of_the_videos_you_share_for_business,delta_confident_to_deliver_an_elevator_pitch,delta_confident_in_managing_businees_finance,delta_understand_the_main_purpose_of_a_business_plan,delta_confident_i_can_prepare_a_complete_and_convincing_business_plan,delta_confident_speaking_in_front_of_audiences
0,2025_IBU-ACADEMY-MIB-C1-01,High,MIB-2025-C1,MIB,2025,26,SABAH,Married,Diploma,Self-Employed,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025_IBU-ACADEMY-MIB-C1-02,High,MIB-2025-C1,MIB,2025,26,JOHOR,Married,SPM,Self-Employed,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025_IBU-ACADEMY-MIB-C1-04,Moderate,MIB-2025-C1,MIB,2025,26,SELANGOR,Divorced,SPM,Self-Employed,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025_IBU-ACADEMY-MIB-C1-05,Moderate,MIB-2025-C1,MIB,2025,26,NEGERI SEMBILAN,Married,Diploma,Self-Employed,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025_IBU-ACADEMY-MIB-C1-06,High,MIB-2025-C1,MIB,2025,26,KUALA LUMPUR,Divorced,SPM,Self-Employed,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# 13.1 Create overall competency change composite

delta_columns = [
    c for c in engineered.columns
    if c.startswith("delta_")
]

print(
    "Individual competency delta features detected:",
    len(delta_columns)
)

for column in delta_columns:
    print("-", column)


def cohort_zscore(series):

    values = pd.to_numeric(
        series,
        errors="coerce"
    )

    sd = values.std(ddof=0)

    if pd.isna(sd) or sd == 0:

        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        values - values.mean()
    ) / sd


if len(delta_columns) > 0:

    z_delta_columns = []

    for column in delta_columns:

        z_name = (
            f"z_{column}"
        )

        engineered[
            z_name
        ] = (
            engineered
            .groupby(
                "Cohort ID"
            )[column]
            .transform(
                cohort_zscore
            )
        )

        z_delta_columns.append(
            z_name
        )


    engineered[
        "overall_competency_change_z"
    ] = (
        engineered[
            z_delta_columns
        ]
        .mean(
            axis=1,
            skipna=True
        )
    )


    engineered[
        "competency_delta_items_available"
    ] = (
        engineered[
            delta_columns
        ]
        .notna()
        .sum(
            axis=1
        )
    )


    engineered.loc[
        engineered[
            "competency_delta_items_available"
        ].eq(0),
        "overall_competency_change_z"
    ] = np.nan


    engineered.drop(
        columns=z_delta_columns,
        inplace=True
    )


    print()
    print(
        "Overall competency composite created."
    )

    print(
        "Available participants:",
        engineered[
            "overall_competency_change_z"
        ]
        .notna()
        .sum()
    )

    print(
        "Coverage %:",
        round(
            engineered[
                "overall_competency_change_z"
            ]
            .notna()
            .mean()
            * 100,
            1
        )
    )


else:

    print(
        "No delta features were created in Notebook 03."
    )

Individual competency delta features detected: 16
- delta_able_to_identify_3_personal_strengths_and_3_weaknesses
- delta_confidence_level_in_your_leadership
- delta_well_know_who_target_customer
- delta_know_the_pain_point_customer_face_that_makes_them_need_productsa
- delta_i_am_confident_with_the_price_i_set
- delta_confidence_level_in_pitching_your_business_idea_in_front_of_judges_or_a_large_audience
- delta_level_of_courage_in_answering_spontaneous_questions_q_a_from_panel_members_or_investors
- delta_comfortable_working_and_sharing_ideas_in_a_team
- delta_confident_with_product_packaging_appearance
- delta_confident_in_the_quality_of_the_photos_you_share_for_business
- delta_confident_in_the_quality_of_the_videos_you_share_for_business
- delta_confident_to_deliver_an_elevator_pitch
- delta_confident_in_managing_businees_finance
- delta_understand_the_main_purpose_of_a_business_plan
- delta_confident_i_can_prepare_a_complete_and_convincing_business_plan
- delta_confident_speaking_i

## 14. Remove Accidental Duplicate Columns


In [24]:
# 14. Check duplicate column names

duplicate_column_names = (
    engineered.columns[
        engineered.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate column names:",
    duplicate_column_names
)

engineered = engineered.loc[
    :,
    ~engineered.columns.duplicated()
].copy()

Duplicate column names: []


## 15. Feature Missingness Audit


In [25]:
# 15. Feature missingness

feature_missingness = pd.DataFrame({
    "Feature": engineered.columns,
    "Available n": [
        engineered[column]
        .notna()
        .sum()
        for column in engineered.columns
    ],
    "Missing n": [
        engineered[column]
        .isna()
        .sum()
        for column in engineered.columns
    ],
    "Missing %": [
        engineered[column]
        .isna()
        .mean()
        * 100
        for column in engineered.columns
    ],
    "Unique Values": [
        engineered[column]
        .nunique(
            dropna=True
        )
        for column in engineered.columns
    ]
})

feature_missingness = (
    feature_missingness
    .sort_values(
        "Missing %",
        ascending=False
    )
)

display(
    feature_missingness
)

,Feature,Available n,Missing n,Missing %,Unique Values
50,delta_understand_the_main_purpose_of_a_busines...,22,151,87.283237,5
48,delta_confident_to_deliver_an_elevator_pitch,22,151,87.283237,5
51,delta_confident_i_can_prepare_a_complete_and_c...,22,151,87.283237,6
43,delta_level_of_courage_in_answering_spontaneou...,23,150,86.705202,5
40,delta_know_the_pain_point_customer_face_that_m...,23,150,86.705202,5
41,delta_i_am_confident_with_the_price_i_set,23,150,86.705202,5
39,delta_well_know_who_target_customer,23,150,86.705202,5
44,delta_comfortable_working_and_sharing_ideas_in...,23,150,86.705202,4
52,delta_confident_speaking_in_front_of_audiences,23,150,86.705202,4
47,delta_confident_in_the_quality_of_the_videos_y...,25,148,85.549133,4


## 16. Candidate Feature Dictionary

In [26]:
# 16. Build candidate feature dictionary

feature_dictionary = []

metadata_features = [
    "Participant ID",
    "Cohort ID",
    "Funder/Program",
    "Year"
]

target_feature = [
    "Performance Trajectory"
]

baseline_business_features = [
    "Sales Before Program",
    "Profit Before Program"
]

engagement_features = [
    "attendance_rate"
]

behavior_feature_names = [
    column
    for column in engineered.columns
    if column.startswith(
        "behavior_"
    )
]

digital_feature_names = [
    "IG_has_pre_binary",
    "IG_followers_pre",
    "FB_has_pre_binary",
    "FB_followers_pre",
    "TT_has_pre_binary",
    "TT_followers_pre",
    "digital_platform_count"
]

formalisation_features = [
    "ssm_registered_binary"
]

delta_feature_names = [
    column
    for column in engineered.columns
    if column.startswith(
        "delta_"
    )
]


for column in engineered.columns:

    if column in metadata_features:
        group = "Metadata"

    elif column in target_feature:
        group = "Target"

    elif column in baseline_business_features:
        group = "Baseline Business"

    elif column in engagement_features:
        group = "Engagement"

    elif column in behavior_feature_names:
        group = "Behavior"

    elif column in digital_feature_names:
        group = "Digital Readiness"

    elif column in formalisation_features:
        group = "Business Formalisation"

    elif column in delta_feature_names:
        group = "Competency Delta"

    else:
        group = "Socio-Demographic"

    feature_dictionary.append({
        "Feature": column,
        "Feature Group": group
    })


feature_dictionary = pd.DataFrame(
    feature_dictionary
)

display(
    feature_dictionary
)

,Feature,Feature Group
0,Participant ID,Metadata
1,Performance Trajectory,Target
2,Cohort ID,Metadata
3,Funder/Program,Metadata
4,Year,Metadata
5,Age,Socio-Demographic
6,State,Socio-Demographic
7,Marital Status,Socio-Demographic
8,Education Level,Socio-Demographic
9,Working Status,Socio-Demographic


## 17. Automated Leakage Scan


In [27]:
# 17. Scan engineered features for possible leakage

leakage_scan_keywords = [
    "month 1",
    "month 2",
    "month 3",
    "total sales",
    "average monthly sales",
    "growth",
    "total profit",
    "post",
    "latest",
    "bonus",
    "performance trajectory"
]

leakage_scan = []

for column in engineered.columns:

    lower = column.lower()

    matched = [
        keyword
        for keyword in leakage_scan_keywords
        if keyword in lower
    ]

    if matched:

        leakage_scan.append({
            "Feature": column,
            "Matched Keyword": ", ".join(
                matched
            )
        })


leakage_scan = pd.DataFrame(
    leakage_scan
)

if len(leakage_scan):

    display(
        leakage_scan
    )

else:

    print(
        "No obvious leakage variables detected by name."
    )

,Feature,Matched Keyword
0,Performance Trajectory,performance trajectory


In [28]:
# 18. Save feature-engineering outputs to Google Drive

ENGINEERED_PATH = (
    OUTPUT_DIR
    / "engineered_features.csv"
)

DELTA_DICTIONARY_PATH = (
    OUTPUT_DIR
    / "competency_delta_dictionary.csv"
)

DELTA_SUMMARY_PATH = (
    OUTPUT_DIR
    / "competency_delta_summary.csv"
)

FEATURE_DICTIONARY_PATH = (
    OUTPUT_DIR
    / "candidate_feature_dictionary.csv"
)

FEATURE_MISSINGNESS_PATH = (
    OUTPUT_DIR
    / "feature_missingness_audit.csv"
)


engineered.to_csv(
    ENGINEERED_PATH,
    index=False
)

delta_feature_dictionary.to_csv(
    DELTA_DICTIONARY_PATH,
    index=False
)

delta_summary.to_csv(
    DELTA_SUMMARY_PATH,
    index=False
)

feature_dictionary.to_csv(
    FEATURE_DICTIONARY_PATH,
    index=False
)

feature_missingness.to_csv(
    FEATURE_MISSINGNESS_PATH,
    index=False
)


print("Saved:")

print("-", ENGINEERED_PATH)
print("-", DELTA_DICTIONARY_PATH)
print("-", DELTA_SUMMARY_PATH)
print("-", FEATURE_DICTIONARY_PATH)
print("-", FEATURE_MISSINGNESS_PATH)

Saved:
- /content/drive/MyDrive/24088950_MAI_Research/outputs/engineered_features.csv
- /content/drive/MyDrive/24088950_MAI_Research/outputs/competency_delta_dictionary.csv
- /content/drive/MyDrive/24088950_MAI_Research/outputs/competency_delta_summary.csv
- /content/drive/MyDrive/24088950_MAI_Research/outputs/candidate_feature_dictionary.csv
- /content/drive/MyDrive/24088950_MAI_Research/outputs/feature_missingness_audit.csv


In [29]:
# 19. Final summary

print("NOTEBOOK 03 SUMMARY")
print()

print(
    "Participants:",
    engineered[
        "Participant ID"
    ].nunique()
)

print(
    "Total columns:",
    engineered.shape[1]
)

print(
    "Competency delta candidates:",
    len(delta_feature_dictionary)
)

print(
    "Behavior features:",
    len(behavior_feature_names)
)

print(
    "Digital features:",
    len([
        column
        for column in digital_feature_names
        if column in engineered.columns
    ])
)

print()
print(
    "Next notebook: 04_Final_Modelling_Dataset.ipynb"
)

NOTEBOOK 03 SUMMARY

Participants: 173
Total columns: 55
Competency delta candidates: 16
Behavior features: 10
Digital features: 7

Next notebook: 04_Final_Modelling_Dataset.ipynb
